In [ ]:
# ============================================================
# Cell 0: 硬件自检（T4 即可；CPU 也能跑通，只是慢很多）
# ============================================================
# 本章要真训一个约 4.8 M 参数的 mini-GPT：3000 步、batch 32、上下文 128。
# T4（15 GB）绰绰有余——显存占用不到 1 GB，训练约 3 分钟。
# 没有 GPU 也能 Run All（代码会自动退回 CPU），但训练那一步要约 1 小时；
# 想先快速走通流程，把 Cell 8 里的 max_steps 调成 500 即可（文本会差一些）。
import sys, platform
import torch

print("Python:", sys.version.split()[0])
print("平台:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}   显存: {props.total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ 当前是 CPU 运行时：能跑通，但 Cell 8 的训练约需 1 小时。")
    print("   Colab 切 GPU：菜单「代码执行程序 → 更改运行时类型 → T4 GPU」")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装依赖
# ============================================================
# %%capture 必须严格在 cell 第一行，把 pip 的安装日志折叠起来。
# torch / matplotlib: 训练与画图要用，Colab 默认已装，故意【不】加 -U 免得换版本。
# transformers:       只在 Cell 3 用一次——拿 Qwen3 的 BBPE tokenizer 和本章的字符级
#                     切法做对照；Qwen3 系列要求 transformers>=4.51，显式锁版本下界。
!pip install -q -U "transformers>=4.51"

In [ ]:
# ============================================================
# Cell 2: 下载 tiny-shakespeare，建一个字符级 tokenizer
# ============================================================
import os, urllib.request
import torch

URL = ("https://raw.githubusercontent.com/karpathy/char-rnn/"
       "master/data/tinyshakespeare/input.txt")
if not os.path.exists("input.txt"):                       # Colab 断线重连后不必重下
    urllib.request.urlretrieve(URL, "input.txt")
text = open("input.txt", encoding="utf-8").read()
print(f"总字符数: {len(text):,}")
print("--- 前 160 个字符 ---")
print(text[:160])

# ---- 字符级词表：语料里出现过的每个字符就是一个 token ----
chars = sorted(set(text))                                 # 排序保证词表可复现（每次跑 id 一致）
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}              # 字符 -> id
itos = {i: ch for i, ch in enumerate(chars)}              # id -> 字符
encode = lambda s: [stoi[c] for c in s]                   # str -> List[int]
decode = lambda ids: "".join(itos[i] for i in ids)        # List[int] -> str
print(f"\n词表大小: {vocab_size}")
print("词表:", repr("".join(chars)))
print("往返验证:", repr(decode(encode("Hello, Shakespeare!"))))

# ---- 整份语料编码成一条长 id 序列，再按 9:1 切出训练 / 验证 ----
data = torch.tensor(encode(text), dtype=torch.long)       # [1115394]，一个 token 一个 int64
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]                 # 按位置前后切，不打乱（保持文本连续）
print(f"\ndata: {tuple(data.shape)} {data.dtype}")
print(f"train: {len(train_data):,} tokens    val: {len(val_data):,} tokens")
print("前 20 个 id:", data[:20].tolist())

In [ ]:
# ============================================================
# Cell 3: 字符级 vs BBPE —— 同一段文本的两种切法
# ============================================================
# from_pretrained 只下载 tokenizer 的几个小文件（几 MB），不下载模型权重。
from transformers import AutoTokenizer

bpe = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")
sample = text[:2000]                                      # 拿开头 2000 个字符做对照
n_char = len(encode(sample))                              # 字符级：一个字符一个 token
# add_special_tokens=False：只切正文，不在两端加 BOS/EOS 之类，数出来的才是纯文本的 token 数。
# len(bpe) 是「基础词表 + 特殊 token」的口径，比 bpe.vocab_size 略大，两个都对、只是数法不同。
n_bpe = len(bpe.encode(sample, add_special_tokens=False))  # BBPE：一个 subword 一个 token
print(f"字符级(本章): 词表 {vocab_size:>6}   2000 字符 -> {n_char:>5} token"
      f"   1 token ≈ {len(sample) / n_char:.2f} 字符")
print(f"Qwen3 BBPE  : 词表 {len(bpe):>6}   2000 字符 -> {n_bpe:>5} token"
      f"   1 token ≈ {len(sample) / n_bpe:.2f} 字符")

line = "First Citizen:\nBefore we proceed"
print("\n同一句话的两种切法（各列前 12 个 token）:")
print("  字符级:", [decode([i]) for i in encode(line)][:12])
print("  BBPE  :", [bpe.decode([i]) for i in bpe.encode(line, add_special_tokens=False)][:12])

# 词表大小直接决定两端那两张大表有多大（按本章的 d_model=256 算）
d_model = 256
print(f"\nembedding / lm_head 各自的参数量（d_model={d_model}）:")
print(f"  字符级: {d_model * vocab_size:>12,}")
print(f"  BBPE  : {d_model * len(bpe):>12,}   <- 比本章整个模型还大")

In [ ]:
# ============================================================
# Cell 4: 从长序列里随机取一批训练样本
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
block_size = 128          # 上下文长度 L：一条样本最多回看 128 个字符
batch_size = 32           # 一批 32 条 -> 一次前向覆盖 32 × 128 = 4096 个位置

def get_batch(split, bs=batch_size, L=block_size):
    """随机取 bs 个起点，各切出一条长度 L 的样本。
    x = 第 i .. i+L-1 个 token（输入）
    y = 第 i+1 .. i+L 个 token（标签，就是 x 整体右移一位）
    返回两个 [bs, L] 的 int64 张量，已搬到 device 上。"""
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - L - 1, (bs,))             # 起点随机；-L-1 保证 y 也取得满
    x = torch.stack([d[i:i + L] for i in ix])             # [bs, L]
    y = torch.stack([d[i + 1:i + L + 1] for i in ix])     # [bs, L]，逐位置右移一位
    return x.to(device), y.to(device)

torch.manual_seed(1337)                                   # 固定种子，本章打印的数字可复现
xb, yb = get_batch("train")
print("device:", device)
print("x:", tuple(xb.shape), xb.dtype, "   y:", tuple(yb.shape), yb.dtype)
print("\n第 0 条样本的前 24 个位置:")
print("  x:", repr(decode(xb[0, :24].tolist())))
print("  y:", repr(decode(yb[0, :24].tolist())), "  <- 整体左移一格，即 x 的下一个字符")
print("\n这 24 个位置其实是 24 条监督（只列前 5 条）:")
for t in range(5):
    ctx = repr(decode(xb[0, :t + 1].tolist()))
    print(f"  看到 {ctx:<22} -> 该预测 {repr(decode([yb[0, t].item()]))}")

In [ ]:
# ============================================================
# Cell 5: 零件——RMSNorm / RoPE / 因果自注意力 / SwiGLU / 一个 Block
# ============================================================
import math
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    """只按均方根缩放、不减均值的归一化。x: [..., d] -> [..., d]。"""
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))    # 每个特征一个可学习增益 γ
        self.eps = eps
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()   # 1/RMS，形状 [..., 1]
        return x * rms * self.weight                                  # 广播回 [..., d]

def build_rope_cache(seq_len, head_dim, base=10000.0):
    """预先算好每个位置 × 每个频率的 cos/sin。返回两个 [seq_len, head_dim] 张量。"""
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))  # [head_dim/2]
    pos = torch.arange(seq_len).float()                                           # [L]
    ang = torch.outer(pos, inv_freq)                                              # [L, head_dim/2]
    cos = torch.cat([ang.cos(), ang.cos()], dim=-1)                               # [L, head_dim]
    sin = torch.cat([ang.sin(), ang.sin()], dim=-1)                               # [L, head_dim]
    return cos, sin

def apply_rope(x, cos, sin):
    """把每个头的向量按所在位置旋转。x: [B, H, L, hd]；cos/sin: [L, hd]。"""
    cos, sin = cos[None, None], sin[None, None]      # [1, 1, L, hd]，便于广播到 batch 和头
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2:]       # 前一半 / 后一半配成旋转对
    rotated = torch.cat([-x2, x1], dim=-1)           # 旋转 90° 的"虚部"
    return x * cos + rotated * sin

class CausalSelfAttention(nn.Module):
    """因果自注意力：H 个 query 头、G 个 KV 头（G=H 即 MHA，G<H 即 GQA），含 RoPE 与上三角掩码。"""
    def __init__(self, d_model, n_heads, n_kv_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)     # 现代 LLM 投影不带 bias
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(n_heads * self.head_dim, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)         # 丢的是注意力权重（哪些位置被看见）
        self.resid_drop = nn.Dropout(dropout)        # 丢的是写回残差流的增量
    def forward(self, x, cos, sin):
        B, L, _ = x.shape
        H, G, hd = self.n_heads, self.n_kv_heads, self.head_dim
        q = self.q_proj(x).view(B, L, H, hd).transpose(1, 2)   # [B, H, L, hd]
        k = self.k_proj(x).view(B, L, G, hd).transpose(1, 2)   # [B, G, L, hd]
        v = self.v_proj(x).view(B, L, G, hd).transpose(1, 2)   # [B, G, L, hd]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)  # 位置信息在这里注入
        if G != H:                                             # GQA：复制凑齐 H 份
            k = k.repeat_interleave(H // G, dim=1)             # -> [B, H, L, hd]
            v = v.repeat_interleave(H // G, dim=1)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(hd)     # [B, H, L, L]
        mask = torch.full((L, L), float("-inf"), device=x.device).triu(1)  # 上三角（含未来）置 -inf
        attn = self.attn_drop((scores + mask).softmax(dim=-1))
        out = attn @ v                                         # [B, H, L, hd]
        out = out.transpose(1, 2).contiguous().view(B, L, H * hd)  # 合头 -> [B, L, d_model]
        return self.resid_drop(self.o_proj(out))

class SwiGLU(nn.Module):
    """SwiGLU 前馈网络：SiLU(gate) ⊙ up，再降维回 d_model。三个无 bias 线性层。"""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class Block(nn.Module):
    """一个 Transformer 层（Pre-LN）：x + Attn(Norm(x))，再 x + FFN(Norm(x))。形状进出守恒。"""
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, n_kv_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)   # 注意力子层：读残差流 -> 算增量 -> 加回
        x = x + self.ffn(self.norm2(x))              # FFN 子层：同上
        return x

In [ ]:
# ============================================================
# Cell 6: 拼出完整的 mini-GPT，并数一遍参数
# ============================================================
class MiniGPT(nn.Module):
    """embedding -> N × Block -> final norm -> lm_head 的 decoder-only 语言模型。
    forward 传了 targets 就顺手把 cross-entropy 也算出来。"""
    def __init__(self, vocab_size, d_model=256, n_layers=6, n_heads=8, n_kv_heads=8,
                 d_ff=None, block_size=128, dropout=0.1):
        super().__init__()
        if d_ff is None:
            d_ff = (int(8 / 3 * d_model) + 7) // 8 * 8    # SwiGLU 三矩阵，取 8/3·d 再对齐到 8 的倍数
        self.block_size = block_size                      # 训练用的最大上下文长度
        self.embed = nn.Embedding(vocab_size, d_model)    # [V, d] 查表
        self.emb_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(d_model, n_heads, n_kv_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)                # 末层归一化（不属于任何 block）
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)   # [d, V] 投回词表
        self.lm_head.weight = self.embed.weight           # weight tying：两端共用同一张表
        cos, sin = build_rope_cache(block_size, d_model // n_heads)
        self.register_buffer("cos", cos)                  # RoPE 表算好就固定，是 buffer 不是 parameter
        self.register_buffer("sin", sin)
        self.apply(self._init_weights)                    # 统一初始化：N(0, 0.02)
        for name, p in self.named_parameters():           # 再把两个"写回残差流"的投影调小
            if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layers))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, ids, targets=None):
        B, L = ids.shape
        assert L <= self.block_size, f"序列 {L} 超过了 block_size={self.block_size}"
        x = self.emb_drop(self.embed(ids))                # [B, L, d]
        cos, sin = self.cos[:L], self.sin[:L]             # 只取前 L 个位置的旋转表
        for blk in self.blocks:
            x = blk(x, cos, sin)                          # 每层形状守恒 [B, L, d]
        logits = self.lm_head(self.final_norm(x))         # [B, L, V]
        if targets is None:                               # 推理：只要 logits
            return logits, None
        # 训练：把 [B, L, V] 拍平成 [B·L, V]、标签拍平成 [B·L]，一次算 B·L 个位置的平均 loss
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

torch.manual_seed(1337)
model = MiniGPT(vocab_size, d_model=256, n_layers=6, n_heads=8, n_kv_heads=8,
                block_size=block_size, dropout=0.1).to(device)

count = lambda m: sum(p.numel() for p in m.parameters())
n_total = count(model)                                    # tying 后 embed 与 lm_head 只算一份
print(f"参数总量: {n_total:,}  ({n_total / 1e6:.2f} M)")
print(f"  embedding(= lm_head, tied): {count(model.embed):>10,}")
print(f"  1 个 Block                : {count(model.blocks[0]):>10,}"
      f"  (attn {count(model.blocks[0].attn):,} + ffn {count(model.blocks[0].ffn):,})")
print(f"  6 个 Block 合计            : {sum(count(b) for b in model.blocks):>10,}")
print(f"  final_norm                : {count(model.final_norm):>10,}")

logits, loss = model(xb, yb)                              # 拿 Cell 4 那批数据前向一次
print(f"\nlogits: {tuple(logits.shape)}   <- [B, L, V]")
print(f"初始 loss: {loss.item():.4f}   随机猜的理论值 ln({vocab_size}) = {math.log(vocab_size):.4f}")

In [ ]:
# ============================================================
# Cell 7: 生成函数 + 训练前的乱码基线
# ============================================================
@torch.no_grad()                                          # 生成不需要梯度
def generate(model, ids, max_new_tokens, temperature=1.0, top_k=None):
    """自回归采样：每步只取最后一个位置的 logits，采一个 token 接到序列末尾。
    ids: [B, L0] 起始上下文（可以只有 1 个 token）；返回 [B, L0 + max_new_tokens]。"""
    was_training = model.training
    model.eval()                                          # 关掉 dropout：推理要用完整的网络
    for _ in range(max_new_tokens):
        ids_cond = ids[:, -model.block_size:]             # 只喂最近 block_size 个（再长模型没见过）
        logits, _ = model(ids_cond)                       # [B, L, V]
        logits = logits[:, -1, :]                         # 只要最后一个位置的预测 -> [B, V]
        if temperature == 0.0:                            # 约定温度 0 = 贪心（顺便避开除零）
            nxt = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature                 # <1 更尖锐、>1 更平坦
            if top_k is not None:                         # top-k：只在概率最高的 k 个里采
                kth = torch.topk(logits, min(top_k, logits.size(-1)))[0][:, [-1]]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            nxt = torch.multinomial(logits.softmax(dim=-1), num_samples=1)   # [B, 1]
        ids = torch.cat([ids, nxt], dim=1)                # 接到末尾，进入下一步
    if was_training:
        model.train()                                     # 还原调用前的模式
    return ids

start = torch.zeros((1, 1), dtype=torch.long, device=device)   # id 0 是换行符，当作空白起点
print("=== 训练前（随机初始化）生成 300 个字符 ===")
print(decode(generate(model, start, 300)[0].tolist()))

In [ ]:
# ============================================================
# Cell 8: 训练循环（T4 约 3 分钟；CPU 约 1 小时——想先走通就把 max_steps 调成 500）
# ============================================================
import time

max_steps = 3000          # 总步数
warmup_steps = 100        # 前 100 步把 lr 从 0 线性升上来
lr_max = 1e-3             # 峰值学习率：模型小、数据小，可以比大模型激进
lr_min = 1e-4             # cosine 退火的下界（= 0.1 × lr_max）
grad_clip = 1.0           # 梯度范数上限
eval_interval = 250       # 每 250 步在 train / val 上各估一次 loss
eval_iters = 20           # 每次估 loss 采 20 个 batch 取平均

opt = torch.optim.AdamW(model.parameters(), lr=lr_max, betas=(0.9, 0.95), weight_decay=0.1)

def lr_at(step):
    """warmup + cosine：先线性升到 lr_max，再余弦退火到 lr_min。"""
    if step < warmup_steps:
        return lr_max * (step + 1) / warmup_steps
    r = (step - warmup_steps) / max(1, max_steps - warmup_steps)      # 0 -> 1
    return lr_min + (lr_max - lr_min) * 0.5 * (1 + math.cos(math.pi * r))

@torch.no_grad()
def estimate_loss():
    """在 train / val 上各采 eval_iters 个 batch 估 loss。
    单个 batch 的 loss 噪声很大，多采几个平均才看得出趋势。"""
    model.eval()                                          # 关 dropout，评估用完整网络
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for i in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()                                         # 记得切回训练模式
    return out

hist_step, hist_loss, hist_gnorm = [], [], []             # 每步的训练 loss / 梯度范数
eval_step, eval_train, eval_val = [], [], []              # 每 eval_interval 步的估计值

model.train()
t0 = time.time()
for step in range(max_steps):
    for g in opt.param_groups:                            # 手写调度：把这一步的 lr 塞进优化器
        g["lr"] = lr_at(step)

    x, y = get_batch("train")                             # 1) 取一批 [32, 128]
    _, loss = model(x, y)                                 # 2) 前向：一次算 32×128 个位置的 loss
    opt.zero_grad(set_to_none=True)                       # 3) 清梯度
    loss.backward()                                       # 4) 反向
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)  # 5) 裁剪，返回裁剪【前】的范数
    opt.step()                                            # 6) 更新

    hist_step.append(step)
    hist_loss.append(loss.item())
    hist_gnorm.append(gnorm.item())
    if step % eval_interval == 0 or step == max_steps - 1:
        e = estimate_loss()
        eval_step.append(step); eval_train.append(e["train"]); eval_val.append(e["val"])
        print(f"step {step:5d} | train {e['train']:.4f} | val {e['val']:.4f} "
              f"| lr {lr_at(step):.2e} | grad_norm {gnorm:5.2f} | {time.time() - t0:6.1f}s")
print(f"训练完成，用时 {time.time() - t0:.1f}s")

In [ ]:
# ============================================================
# Cell 9: 训练曲线 —— loss 与梯度范数
# ============================================================
# matplotlib 的默认字体不含中文字形，所有图内文字一律用英文，避免渲染成方框。
import matplotlib.pyplot as plt

def smooth(xs, k=50):
    """长度为 k 的滑动平均：单步 loss 抖得厉害，平滑一下才看得出趋势。"""
    out, s = [], 0.0
    for i, v in enumerate(xs):
        s += v
        if i >= k:
            s -= xs[i - k]
        out.append(s / min(i + 1, k))
    return out

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_step, hist_loss, alpha=0.25, color="tab:blue", label="train loss (per step)")
axes[0].plot(hist_step, smooth(hist_loss), color="tab:blue", label="train loss (smoothed)")
axes[0].plot(eval_step, eval_train, "o-", color="tab:green", label="train loss (eval)")
axes[0].plot(eval_step, eval_val, "s-", color="tab:red", label="val loss (eval)")
axes[0].axhline(math.log(vocab_size), color="gray", ls=":", label="random baseline ln(65)")
axes[0].set_xlabel("step"); axes[0].set_ylabel("cross-entropy (nats / token)")
axes[0].set_title("mini-GPT on tiny-shakespeare"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist_step, hist_gnorm, alpha=0.6, color="tab:purple")
axes[1].axhline(grad_clip, color="tab:red", ls="--", label=f"clip threshold = {grad_clip}")
axes[1].set_yscale("log")
axes[1].set_xlabel("step"); axes[1].set_ylabel("grad norm (before clipping)")
axes[1].set_title("gradient norm"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"末次估计    : train {eval_train[-1]:.4f}   val {eval_val[-1]:.4f}")
print(f"对应困惑度  : train {math.exp(eval_train[-1]):.2f}   val {math.exp(eval_val[-1]):.2f}")
print(f"随机猜的基线: loss ln({vocab_size}) = {math.log(vocab_size):.4f}，困惑度 {vocab_size}")
print(f"被裁剪的步数: {sum(g > grad_clip for g in hist_gnorm)} / {len(hist_gnorm)}")

In [ ]:
# ============================================================
# Cell 10: 训练后生成 —— 采样旋钮对照
# ============================================================
torch.manual_seed(2024)                                   # 固定采样随机性，便于复现下面的输出
prompt = "ROMEO:"
pid = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

print("=== A. 贪心 temperature=0（从 ROMEO: 起步）===")
print(decode(generate(model, pid, 400, temperature=0.0)[0].tolist()))

print("\n=== B. temperature=0.8 + top_k=40（常用档）===")
print(decode(generate(model, start, 400, temperature=0.8, top_k=40)[0].tolist()))

print("\n=== C. temperature=1.5（过热）===")
print(decode(generate(model, start, 200, temperature=1.5)[0].tolist()))

print("\n=== D. 同一个开头改用采样，和 A 对照 ===")
print(decode(generate(model, pid, 300, temperature=0.8, top_k=40)[0].tolist()))

In [ ]:
# ============================================================
# Cell 11: 存档与再加载 —— 权重之外还得存词表和结构超参
# ============================================================
ckpt = {
    "model_state": model.state_dict(),                    # 权重本体
    # 结构超参：必须和训练时完全一致，否则 load_state_dict 会因形状对不上而报错。
    # d_ff 这里没存，是因为它由 MiniGPT 按 d_model 自动算；若你手动指定过 d_ff，记得一并存。
    "config": dict(vocab_size=vocab_size, d_model=256, n_layers=6, n_heads=8,
                   n_kv_heads=8, block_size=block_size, dropout=0.1),
    "stoi": stoi, "itos": itos,                           # 词表：没有它 id 无法还原成字符
}
torch.save(ckpt, "mini_gpt.pt")
print("已保存 mini_gpt.pt，大小 %.2f MB" % (os.path.getsize("mini_gpt.pt") / 1024**2))

# 假装是新开的 session：只有这个文件，从头把模型建回来
ckpt = torch.load("mini_gpt.pt", map_location=device, weights_only=False)
model2 = MiniGPT(**ckpt["config"]).to(device)
model2.load_state_dict(ckpt["model_state"])
print("重建完成，参数是否逐个相等:",
      all(torch.equal(a, b) for a, b in zip(model.state_dict().values(),
                                            model2.state_dict().values())))

torch.manual_seed(7)
out1 = decode(generate(model, start, 120, temperature=0.8, top_k=40)[0].tolist())
torch.manual_seed(7)
out2 = decode(generate(model2, start, 120, temperature=0.8, top_k=40)[0].tolist())
print("同一随机种子下两个模型的输出是否一致:", out1 == out2)

In [ ]:
# ============================================================
# Cell 12: 生成慢在哪 —— 每生成一个 token 都要把整个前缀重算一遍
# ============================================================
model.eval()
lens, per_token_ms = [], []
for n_ctx in [1, 8, 16, 32, 64, 96, 128]:
    ids = torch.randint(0, vocab_size, (1, n_ctx), device=device)
    with torch.no_grad():
        for _ in range(3):                                # 预热，别把首次调用的初始化开销算进去
            model(ids)
        if device == "cuda":
            torch.cuda.synchronize()                      # GPU 是异步的，计时前先同步
        t0 = time.time()
        for _ in range(20):                               # 重复 20 次取平均，减少抖动
            model(ids)
        if device == "cuda":
            torch.cuda.synchronize()
        dt = (time.time() - t0) / 20 * 1000               # 毫秒
    lens.append(n_ctx); per_token_ms.append(dt)
    print(f"前缀 {n_ctx:4d} 个 token -> 再生成 1 个 token 需要一次前向: {dt:7.2f} ms")

plt.figure(figsize=(6.5, 4))
plt.plot(lens, per_token_ms, "o-", color="tab:orange")
plt.xlabel("prefix length (tokens already generated)")
plt.ylabel("time for one more token (ms)")
plt.title("Cost of one generated token (recomputing the whole prefix)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

n = block_size
print(f"\n连续生成 {n} 个 token（前缀从 1 个位置长到 {n} 个）：累计前向了 "
      f"{sum(range(1, n + 1)):,} 个位置，其中真正新出现的只有 {n} 个。")
print(f"也就是说约 {100 * (1 - n / sum(range(1, n + 1))):.1f}% 的计算是在重复上一步已经算过的东西。")